In [1]:
#Edge detection
import cv2 as cv
import numpy as np

def nothing(x):
    pass

# 1. THIẾT LẬP THANH TRƯỢT
cv.namedWindow('Canny Tuning')
cv.createTrackbar('Min Val', 'Canny Tuning', 100, 255, nothing)
cv.createTrackbar('Max Val', 'Canny Tuning', 200, 255, nothing)

# 2. ĐỌC VIDEO (Hoặc ảnh)
path = r'F:\lapTrinhPy\test.mp4'
cap = cv.VideoCapture(path)

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv.CAP_PROP_POS_FRAMES, 0) # Tua lại video
        continue
    
    # Resize cho nhẹ máy
    frame = cv.resize(frame, (640, 360))

    # ==========================================
    # QUY TRÌNH XỬ LÝ CHUẨN (Preprocessing)
    # ==========================================
    
    # Bước 1: Chuyển sang ảnh xám (Bắt buộc)
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Bước 2: Làm mờ Gaussian (Quan trọng để Canny không bắt nhiễu)
    # Kích thước (5, 5) là chuẩn bài
    blur = cv.GaussianBlur(gray, (5, 5), 0)

    # Bước 3: Lấy giá trị từ thanh trượt
    min_val = cv.getTrackbarPos('Min Val', 'Canny Tuning')
    max_val = cv.getTrackbarPos('Max Val', 'Canny Tuning')

    # Bước 4: Áp dụng Canny
    edges = cv.Canny(blur, min_val, max_val)

    # ==========================================
    # HIỂN THỊ KẾT QUẢ
    # ==========================================
    # Hiển thị ảnh gốc và ảnh biên cạnh nhau
    cv.imshow('Original Video', frame)
    cv.imshow('Canny Edges (Net ve)', edges)

    if cv.waitKey(50) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

In [ ]:
#Morphology
import cv2 as cv
import numpy as np

# 1. TẠO ẢNH GIẢ LẬP (Để dễ hình dung nhiễu và lỗ thủng)
# Tạo nền đen
img = np.zeros((400, 600), dtype='uint8')
# Vẽ chữ 'J' to màu trắng (Vật thể chính)
cv.putText(img, 'J', (180, 280), cv.FONT_HERSHEY_SIMPLEX, 10, (255), 20)

# Tạo nhiễu (Giả sử ảnh bị bẩn)
# - Nhiễu trắng bên ngoài (cần xóa bằng Erosion/Opening)
noise_white = np.random.randint(0, 2, (400, 600)) * 255 # Tạo đốm trắng ngẫu nhiên
noise_white = noise_white.astype('uint8')
# Lọc bớt nhiễu cho đỡ dày đặc (Code mẹo để tạo nhiễu thưa)
mask_noise = np.random.rand(400, 600) > 0.98 
img[mask_noise] = 255 # Gán đốm trắng

# - Lỗ thủng đen bên trong chữ J (cần vá bằng Dilation/Closing)
img[200:210, 200:300] = 0 # Tạo vết nứt ngang chữ J

cv.imshow('1. Anh Goc (Bi Nhieu + Nut)', img)

# ==========================================
# CHUẨN BỊ KERNEL (Phần tử cấu trúc)
# ==========================================
# Tạo một ma trận vuông 5x5 toàn số 1
# Kernel càng to (ví dụ 9x9) thì Co/Giãn càng mạnh
kernel = np.ones((5, 5), np.uint8)

# ==========================================
# 2. PHÉP CO (EROSION)
# ==========================================
# iterations: Số lần lặp lại (Co 1 lần hay co 2, 3 lần)
erosion = cv.erode(img, kernel, iterations=1)

# ==========================================
# 3. PHÉP GIÃN (DILATION)
# ==========================================
dilation = cv.dilate(img, kernel, iterations=1)

# ==========================================
# 4. OPENING (Chuyên trị nhiễu nền)
# ==========================================
# Hàm morphologyEx tự động làm: Erode -> Dilate
opening = cv.morphologyEx(img, cv.MORPH_OPEN, kernel)

# ==========================================
# 5. CLOSING (Chuyên trị lỗ thủng)
# ==========================================
# Hàm morphologyEx tự động làm: Dilate -> Erode
closing = cv.morphologyEx(img, cv.MORPH_CLOSE, kernel)

# ==========================================
# HIỂN THỊ KẾT QUẢ
# ==========================================
cv.imshow('2. Erosion (Mat nhieu, nhung chu bi gay)', erosion)
cv.imshow('3. Dilation (Lien mach, nhung nhieu to ra)', dilation)
cv.imshow('4. Opening (Xoa sach nhieu trang)', opening)
cv.imshow('5. Closing (Va lanh vet nut)', closing)

print("👉 Hãy so sánh Opening và Closing để thấy sự khác biệt.")
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
#Threhold, Adaptive Threholding
import cv2 as cv
import numpy as np

# 1. ĐỌC ẢNH (Bắt buộc phải chuyển về ảnh XÁM trước khi Threshold)
path = r'D:\Test code CV\Photos\do_bao_ho.jpg'
# Dùng cờ 0 để đọc thẳng thành ảnh xám (Grayscale)
img = cv.imread(path, 0) 

if img is None:
    print("❌ Lỗi: Không thấy ảnh!")
    exit()

# Resize cho dễ nhìn nếu ảnh quá to
img = cv.resize(img, (600, 400))

# ==========================================
# CÁCH 1: SIMPLE THRESHOLD (Ngưỡng tĩnh)
# ==========================================
# Cú pháp: threshold(ảnh_xám, ngưỡng, giá_trị_max, kiểu_phân_loại)
# 127 là ngưỡng trung bình.
ret, thresh_simple = cv.threshold(img, 127, 255, cv.THRESH_BINARY)

# Mẹo: THRESH_BINARY_INV là đảo ngược (Nền trắng, vật đen)
ret, thresh_inv = cv.threshold(img, 127, 255, cv.THRESH_BINARY_INV)

# ==========================================
# CÁCH 2: ADAPTIVE THRESHOLD (Ngưỡng thích ứng) - QUAN TRỌNG
# ==========================================
# Cú pháp: adaptiveThreshold(ảnh, max_val, phương_pháp, kiểu_ngưỡng, block_size, C)
# - block_size (11): Kích thước vùng lân cận (phải là số lẻ: 11, 13, 15...)
# - C (2): Hằng số trừ đi để tinh chỉnh (giúp khử nhiễu).

# a. Thích ứng theo Trung bình (Mean)
thresh_mean = cv.adaptiveThreshold(img, 255, cv.ADAPTIVE_THRESH_MEAN_C, \
                                   cv.THRESH_BINARY, 11, 2)

# b. Thích ứng theo Gaussian (Khuyên dùng cho Đồ án)
thresh_gauss = cv.adaptiveThreshold(img, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, \
                                    cv.THRESH_BINARY, 11, 2)

# ==========================================
# HIỂN THỊ SO SÁNH
# ==========================================
cv.imshow('1. Anh Xam Goc', img)
cv.imshow('2. Simple Threshold (127)', thresh_simple)
cv.imshow('3. Adaptive Mean', thresh_mean)
cv.imshow('4. Adaptive Gaussian (Xin nhat)', thresh_gauss)

print("👉 Nhận xét: Adaptive Gaussian thường cho nét mảnh và rõ nhất.")
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
#Blur, Gaussian, Median
import cv2 as cv
import numpy as np

# 1. CẤU HÌNH
path = r'D:\Test code CV\Videos\Cong_truong_1.mp4'
cap = cv.VideoCapture(path)

if not cap.isOpened():
    print("❌ Lỗi: Không mở được video!")
    exit()

# Lấy FPS để tính delay chuẩn
fps = cap.get(cv.CAP_PROP_FPS)
delay_chuan = int(1000 / fps)

# Kích thước hạt nhân (Kernel Size)
# Tôi để số to (15) cho bạn dễ nhìn sự khác biệt.
# Vào thực tế đồ án thì chỉ nên để 3 hoặc 5 thôi.
k_size = 5 

print("👉 Bấm phím 'p' hoặc 'SPACE' để Tạm dừng/Tiếp tục.")
print("👉 Bấm phím 'q' để Thoát.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Hết video, tua lại từ đầu...")
        cap.set(cv.CAP_PROP_POS_FRAMES, 0)
        continue

    # Resize nhỏ lại để hiển thị vừa màn hình (mỗi cửa sổ 480x360)
    frame_small = cv.resize(frame, (480, 360))

    
    # ÁP DỤNG 3 KỸ THUẬT LÀM MỜ
    
    
    # 1. Average Blur (Mờ trung bình) -> Nhòe đều, mất nét
    blur_avg = cv.blur(frame_small, (k_size, k_size))

    # 2. Gaussian Blur (Mờ Gaussian) -> Mờ mịn, tự nhiên hơn
    blur_gauss = cv.GaussianBlur(frame_small, (k_size, k_size), 0)

    # 3. Median Blur (Mờ trung vị) -> Tranh sơn dầu, mất chi tiết nhỏ
    # Lưu ý: Median chỉ nhận 1 số nguyên (k), không phải tuple (k,k)
    blur_median = cv.medianBlur(frame_small, k_size)

   
    # HIỂN THỊ 4 CỬA SỔ
    
    # Sắp xếp vị trí cửa sổ cho gọn (Nếu máy bạn hỗ trợ)
    cv.imshow('1. VIDEO GOC (Original)', frame_small)
    cv.moveWindow('1. VIDEO GOC (Original)', 0, 0) # Góc trên trái

    cv.imshow('2. Average Blur (Nhoe)', blur_avg)
    cv.moveWindow('2. Average Blur (Nhoe)', 500, 0) # Góc trên phải

    cv.imshow('3. Gaussian Blur (Min)', blur_gauss)
    cv.moveWindow('3. Gaussian Blur (Min)', 0, 400) # Góc dưới trái

    cv.imshow('4. Median Blur (Khu hat)', blur_median)
    cv.moveWindow('4. Median Blur (Khu hat)', 500, 400) # Góc dưới phải

    
    # XỬ LÝ PHÍM BẤM (PAUSE / QUIT)
    
    key = cv.waitKey(delay_chuan) & 0xFF

    # Nếu bấm 'q' thì thoát
    if key == ord('q'):
        break
    
    # Nếu bấm 'p' hoặc 'dấu cách' thì PAUSE
    elif key == ord('p') or key == 32: # 32 là mã ASCII của Space
        print("⏸ Đang tạm dừng. Bấm phím bất kỳ để tiếp tục...")
        cv.waitKey(0) # Chờ vô tận cho đến khi bấm phím bất kỳ

cap.release()
cv.destroyAllWindows()

In [ ]:
#Resize, Crop, Rotate
import cv2 as cv
import numpy as np

# 1. ĐỌC ẢNH ĐẦU VÀO
path = r'D:\Test code CV\Photos\do_bao_ho.jpg'
img = cv.imread(path)

if img is None:
    print("❌ Lỗi: Không tìm thấy ảnh đầu vào!")
    exit()

print(f"📏 Kích thước gốc: {img.shape}") 
# Kết quả thường là (Height, Width, Channel) - VD: (1080, 1920, 3)


# KỸ THUẬT 1: RESIZE (Thay đổi kích thước)

# Lưu ý quan trọng: OpenCV quy định thứ tự là (Width, Height)
# Resize về kích thước cố định (600x400)
img_resize = cv.resize(img, (600, 400))

# Resize theo tỉ lệ (Ví dụ: giảm 50%)
# fx=0.5 (Chiều ngang còn 50%), fy=0.5 (Chiều dọc còn 50%)
img_scale = cv.resize(img, None, fx=0.5, fy=0.5)



# KỸ THUẬT 2: CROP (Cắt vùng quan tâm - ROI)

# Sử dụng cơ chế Slicing của NumPy Array
# Cú pháp NumPy: img[Start_Y:End_Y, Start_X:End_X]
# Cắt lấy vùng giữa ảnh (Bỏ bớt trời và đất)
# Từ dòng 200 đến 800, Từ cột 500 đến 1500
img_crop = img[200:800, 500:1500]



# KỸ THUẬT 3: ROTATE (Xoay ảnh)

# Xoay quanh tâm ảnh một góc 45 độ
def rotate_image(image, angle):
    # Lấy chiều cao, chiều rộng
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2) # Tìm tâm ảnh

    # Tạo ma trận xoay (Rotation Matrix)
    M = cv.getRotationMatrix2D(center, angle, 1.0) # 1.0 là tỉ lệ scale (giữ nguyên)

    # Thực hiện biến đổi
    rotated = cv.warpAffine(image, M, (w, h))
    return rotated

img_rotate = rotate_image(img, 45) # Xoay 45 độ



# HIỂN THỊ KẾT QUẢ

cv.imshow('1. Anh Goc', img)
cv.imshow('2. Resize (600x400)', img_resize)
cv.imshow('3. Crop (ROI)', img_crop)
cv.imshow('4. Rotate 45 do', img_rotate)

print("✅ Đã xử lý xong. Bấm phím bất kỳ để thoát.")
cv.waitKey(0)
cv.destroyAllWindows()

In [5]:
#Đọc&Ghi videos
import cv2 as cv
import sys

#Phần 1: Thiết lập đường dẫn
path_input = r'F:\lapTrinhPy\test.mp4'
path_output = r'F:\lapTrinhPy\Ket_qua_ctr_1.avi'

# đọc video
video = cv.VideoCapture(path_input)

#kiểm tra dữ liệu đầu vào
if not video.isOpened():
    print('Không tìm thấy video!')
    print(f'Kiêm tra lại xem file {path_input} có đuôi mp4 chưa')
    sys.exit()
else:
    print ('Đã tìm thấy video')

# Phần 2: Thông số kỹ thuật của video
w = int (video.get(cv.CAP_PROP_FRAME_WIDTH)) #chiều rộng
h = int (video.get(cv.CAP_PROP_FRAME_HEIGHT)) #chiều dài
fps = int (video.get(cv.CAP_PROP_FPS))
print(f'Kích thước video {w}x{h}')
print(f'Tốc độ khung hình: {fps} FPS')


#Phần 3: Khởi tạo máy ghi
fourcc = cv.VideoWriter_fourcc(*'MJPG') 
#Đây là mã gồm 4 ký tự dùng để xác định định dạng nén video (codec)
#MJPG: là dạng codec Motion-JEPG
#Nếu muốn lưu file dưới dạng MP4: thay "mp4v" vào thay cho "MJPG" hoặc "XVID"
out = cv.VideoWriter(path_output, fourcc, fps, (w, h))
print(f'Đang ghi hình file: {path_input}')
print('Bấm q để tắt video')

#Phần 4: Vòng lặp xử lý
while True:
    #Đọc từng khung hình:
    isTrue, frame = video.read()

    #Kiểm tra: Nếu hết video (isTrue = False) thì thoát
    if not isTrue:
        print('Đã chạy hết video')
        break

    #Nhét khung hình hiện tại vào file kết quả
    out.write(frame)

    #Hiển thị video
    cv.imshow('Giám sát công trình', frame)

    #Điều khiển tốc độ 
    delay = int(1000/fps)

    if cv.waitKey(delay) & 0xFF == ord('q'):
        print('Đã tắt video')
        break


#Phần 5: Kết thúc
video.release()
out.release() #lưu file 
cv.destroyAllWindows()

Đã tìm thấy video
Kích thước video 1920x1080
Tốc độ khung hình: 23 FPS
Đang ghi hình file: F:\lapTrinhPy\test.mp4
Bấm q để tắt video
Đã tắt video


In [4]:
#Đọc&Ghi Ảnh (có chỉnh màu)
import cv2 as cv 
#Mở thu viện OpenCV, "as" gán định danh là "cv" tên viết tắt quy chuẩn (không gán cũng được)
import sys
#Hộp dụng cụ có sẵn của Python - system: hệ thống - dùng để xử lý lỗi

path_input = r'D:\Test code CV\Photos\do_bao_ho.jpg'
path_output = r'D:\Test code CV\Photos\do_bao_ho_xam.jpg'
#Đặt biến cho input và output 
#path cài đặt đường đẫn đến file và ảnh cần thực hiện nhiệm vụ

#Phần 1: Đọc ảnh

img = cv.imread(path_input)
# Hàm imread dùng để đọc ảnh và chuyển đổi sang ma trận số
# Gán biến img luôn

#Kiểm tra dữ liệu đầu vào
if img is None:
    print(f'Lỗi không thể tìm thấy ảnh tại: {path_input}') #khi gán gia trị trong { } có thể thay đổi thì ta thêm "f"
    print('Kiểm tra lại: 1. File đã đúng chưa? 2. Đuôi file (.jpg/.png)?')
    sys.exit() #dừng chương trình ngay lập tức
else:
    print(f'Đã tìm thấy ảnh')
    print(f'Kích thước ảnh: {img.shape}') 


#Phần 2: Xử lý sơ bộ:
gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY) #thay màu ảnh sang màu xám

#Phần 3: Hiển thị

cv.imshow('Anh goc', img)
# hiển thị ảnh gốc chưa qua chỉnh màu
cv.imshow('Anh da chinh', gray_img)
# hiển thị ảnh đã qua chỉnh màu

#Phần 4: Ghi ảnh 
#Nghĩa là lưu lại ảnh vào ổ mình muốn
cv.imwrite(path_output, gray_img)
print(f'Đã lưu ảnh xám tại: {path_output}')

#Kết thúc chương trình

cv.waitKey(0) #chờ cho đến khi người dùng bấm 1 nút bất kỳ - kết thúc chương trình 
cv.destroyAllWindows() #Giải phòng bộ nhớ và đóng các cửa sổ

Lỗi không thể tìm thấy ảnh tại: D:\Test code CV\Photos\do_bao_ho.jpg
Kiểm tra lại: 1. File đã đúng chưa? 2. Đuôi file (.jpg/.png)?


SystemExit: 

C:\Users\chn\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
